[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-02-tracking-experiments.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Tracking Experiments with Runs, Params, and Metrics
**certified-journeys / mlflow-certified** · Day 2 · Experiment Management

> **Goal for today:** Create named experiments, log epoch-by-epoch metrics as time series, tag runs for filtering, and compare multiple runs programmatically.

In [ ]:
%pip install -q mlflow scikit-learn pandas matplotlib numpy

## Step 1 · Named Experiments with `set_experiment`

By default every run lands in the **Default** experiment (id `"0"`). Named experiments let you group related runs together and filter the UI to just the work you care about.

| API | Behaviour |
|---|---|
| `mlflow.set_experiment(name)` | Creates the experiment if it doesn't exist; sets it active for the process |
| `mlflow.get_experiment_by_name(name)` | Returns the `Experiment` object (id, artifact_location, …) |
| `mlflow.create_experiment(name, artifact_location=...)` | Explicit creation — lets you set a custom artifact root |

**Best practice:** call `set_experiment` at the very top of every script, before any `start_run` calls. Orphaned runs in Default are hard to find later.

Experiment names are strings; keep them short and meaningful: `iris-lr-sweep`, `fraud-detection-v2`.

In [ ]:
import mlflow

# Create (or reuse) a named experiment
experiment_name = "iris-classification"
mlflow.set_experiment(experiment_name)

# Inspect the experiment metadata
exp = mlflow.get_experiment_by_name(experiment_name)
print(f"Experiment name     : {exp.name}")
print(f"Experiment ID       : {exp.experiment_id}")
print(f"Artifact location   : {exp.artifact_location}")
print(f"Lifecycle stage     : {exp.lifecycle_stage}")

### What just happened?
- **`set_experiment`** is idempotent — safe to call every time the script starts; it reuses an existing experiment by name.
- **`experiment_id`** is what MLflow uses internally; you will need it for `search_runs` and `MlflowClient` calls.
- **`artifact_location`** shows where artifact files will be written — by default a `mlruns/<id>/` directory.
- **`lifecycle_stage`** will be `active` until you delete the experiment, at which point it becomes `deleted`.

## Step 2 · Explicit Run Lifecycle: `start_run` / `end_run`

The `with mlflow.start_run():` pattern is convenient but not always practical — for example when runs span multiple cells in a notebook or when you want to control the run from a wrapper function.

You can manage run lifecycle explicitly:

```python
run = mlflow.start_run(run_name="my-run")   # opens the run
# ... log things ...
mlflow.end_run()                             # closes it; status = FINISHED

# To mark a run as failed:
mlflow.end_run(status="FAILED")
```

**Active run** — at most one run can be active per process at a time. `mlflow.active_run()` returns it (or `None`).

Use the `with` pattern whenever possible; use explicit calls when you need fine-grained control.

In [ ]:
import mlflow
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-classification")  # active experiment for this session

# --- Explicit lifecycle ---
active_run = mlflow.start_run(run_name="explicit-lifecycle-demo")
print(f"Run opened: {active_run.info.run_id[:8]}")
print(f"Active run: {mlflow.active_run().info.run_id[:8]}")

mlflow.log_param("C", 0.5)
mlflow.log_param("solver", "lbfgs")

clf = LogisticRegression(C=0.5, max_iter=200, solver="lbfgs")
clf.fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))
mlflow.log_metric("test_accuracy", acc)

# Explicitly close the run
mlflow.end_run()  # status defaults to FINISHED

print(f"Active run after end_run: {mlflow.active_run()}")
print(f"Test accuracy logged    : {acc:.4f}")

### What just happened?
- **`mlflow.start_run()`** returns the `ActiveRun` object and sets it as the process-level active run.
- **`mlflow.active_run()`** returns `None` after `end_run()` — safe to call anytime to check state.
- If you forget `end_run()` and call `start_run()` again, MLflow raises a warning and nests the run.
- **In notebooks**, always call `end_run()` at the bottom of a cell that opens a run, or use the `with` pattern.

## Step 3 · Time-Series Metrics: Logging Per-Epoch

The real power of MLflow metrics is their **step** parameter. By logging a metric at each training step you get a time-series chart in the UI:

```python
mlflow.log_metric("loss", value, step=epoch)
```

Rules for `step`:
- Must be a non-negative integer
- Does **not** need to be strictly increasing (you can log multiple metrics at the same step)
- Defaults to `0` if omitted — all points stack on step 0 (single-value metric)

Use `mlflow.log_metrics({...}, step=epoch)` to log multiple metrics atomically at the same step.

In [ ]:
import mlflow
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-classification")

# Simulate epoch-by-epoch training by partially fitting an MLP
N_EPOCHS = 20

with mlflow.start_run(run_name="mlp-epoch-tracking") as run:
    mlflow.log_params({
        "hidden_layer_sizes": "(64, 32)",
        "learning_rate_init": 0.01,
        "max_iter": 1,            # one epoch at a time via partial_fit
        "n_epochs": N_EPOCHS,
    })

    # MLPClassifier with partial_fit simulates epoch-by-epoch training
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        learning_rate_init=0.01,
        max_iter=1,
        warm_start=True,   # keep weights between partial_fit calls
        random_state=42,
    )

    classes = np.unique(y_train)

    for epoch in range(N_EPOCHS):
        mlp.partial_fit(X_train, y_train, classes=classes)

        train_acc  = accuracy_score(y_train, mlp.predict(X_train))
        test_acc   = accuracy_score(y_test,  mlp.predict(X_test))
        train_loss = log_loss(y_train, mlp.predict_proba(X_train))

        # Log all three metrics at the same step
        mlflow.log_metrics({
            "train_accuracy" : train_acc,
            "test_accuracy"  : test_acc,
            "train_loss"     : train_loss,
        }, step=epoch)

    print(f"Run ID          : {run.info.run_id[:8]}")
    print(f"Final test acc  : {test_acc:.4f}")
    print(f"Epochs logged   : {N_EPOCHS} steps per metric")

### What just happened?
- **`step=epoch`** creates 20 data points per metric — the UI renders these as line charts.
- **`log_metrics({...}, step=epoch)`** batches all three metrics at the same step in a single call.
- **`warm_start=True` + `partial_fit`** lets us simulate epoch loops with sklearn (normally used with frameworks like PyTorch that have native epoch loops).
- In the UI → Metrics tab, you can overlay `train_accuracy` and `test_accuracy` to spot overfitting.

## Step 4 · Tagging Runs

**Tags** are free-form key-value string metadata you attach to a run. Unlike params (fixed at log time) and metrics (numeric), tags can be set and updated at any point during or after a run.

```python
mlflow.set_tag("team", "nlp-squad")                    # single tag
mlflow.set_tags({"env": "staging", "baseline": "true"}) # batch
```

**MLflow system tags** (prefixed with `mlflow.`) are set automatically:

| Tag | Value |
|---|---|
| `mlflow.source.name` | Script filename or notebook URL |
| `mlflow.user` | OS username |
| `mlflow.runName` | Value passed to `run_name=` |

**Common custom tag patterns:**
- `model_type`: `logistic_regression`, `mlp`, `xgboost`
- `dataset_version`: `v1.2`, `2024-06-01`
- `is_baseline`: `true` / `false`
- `owner`: `alice`

In [ ]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-classification")

with mlflow.start_run(run_name="random-forest-tagged") as run:
    mlflow.log_params({"n_estimators": 100, "max_depth": 5, "random_state": 42})

    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    mlflow.log_metric("test_accuracy", acc)

    # Set informational tags — searchable in the UI and via search_runs
    mlflow.set_tags({
        "model_type"     : "random_forest",
        "dataset"        : "iris",
        "dataset_version": "sklearn-builtin",
        "is_baseline"    : "false",
        "owner"          : "day2-notebook",
    })

    print(f"Run ID   : {run.info.run_id[:8]}")
    print(f"Test acc : {acc:.4f}")
    print(f"Tags set : model_type, dataset, dataset_version, is_baseline, owner")

### What just happened?
- **`set_tags({...})`** records all five tags atomically on the run.
- In the UI you can filter the run list with `tags.model_type = 'random_forest'` to narrow results.
- **Tags survive across `end_run()`** — you can add tags to a finished run via `MlflowClient.set_tag(run_id, key, value)`.
- **`is_baseline`** as a tag is a common pattern for marking the reference run in a comparison.

## Step 5 · Filtering Runs by Tag and Comparing Results

The **`search_runs`** function accepts a `filter_string` using SQL-like syntax:

```python
# Filter by tag
mlflow.search_runs(filter_string="tags.model_type = 'random_forest'")

# Filter by metric threshold
mlflow.search_runs(filter_string="metrics.test_accuracy > 0.9")

# Combine conditions
mlflow.search_runs(
    filter_string="tags.model_type = 'logistic_regression' AND metrics.test_accuracy > 0.9"
)
```

`mlflow.search_runs` (module-level) returns a **Pandas DataFrame** directly — convenient for analysis.

`MlflowClient.search_runs` returns a list of `Run` objects — use this when you need full run metadata.

In [ ]:
import mlflow
import pandas as pd

# Get the experiment ID for the named experiment
exp = mlflow.get_experiment_by_name("iris-classification")
exp_id = exp.experiment_id

# --- Retrieve all finished runs as a DataFrame ---
df_all = mlflow.search_runs(
    experiment_ids=[exp_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.test_accuracy DESC"],
)

# Select only the columns we care about for the comparison table
cols = [c for c in df_all.columns if c in [
    "run_id", "tags.mlflow.runName",
    "params.C", "params.n_estimators", "params.hidden_layer_sizes",
    "metrics.test_accuracy", "metrics.train_accuracy",
    "tags.model_type",
]]

df_compare = df_all[cols].copy()
df_compare["run_id"] = df_compare["run_id"].str[:8]  # truncate

print("=== Run Comparison ===")
with pd.option_context('display.max_columns', None, 'display.width', 120):
    print(df_compare.to_string(index=False))

# --- Filter: only runs where test_accuracy > 0.9 ---
df_top = mlflow.search_runs(
    experiment_ids=[exp_id],
    filter_string="metrics.test_accuracy > 0.9",
    order_by=["metrics.test_accuracy DESC"],
)
print(f"\nRuns with test_accuracy > 0.90: {len(df_top)}")
if len(df_top):
    print(df_top[["run_id", "tags.mlflow.runName", "metrics.test_accuracy"]].to_string(index=False))

### What just happened?
- **`mlflow.search_runs`** (module-level) returns a DataFrame where every column is prefixed: `params.X`, `metrics.X`, `tags.X`.
- **`filter_string`** supports `=`, `!=`, `>`, `<`, `>=`, `<=`, `LIKE`, `ILIKE`, and `AND`/`OR` — same syntax as the UI filter bar.
- The **comparison table** here mirrors the "Compare Runs" feature in the UI where you tick two runs and click Compare.
- **`order_by=["metrics.test_accuracy DESC"]`** returns the best run first — useful for automated best-model selection.

## Step 6 · Parent / Child Runs (Nested Runs)

When running a hyperparameter sweep you often want to group all child runs under a single parent. MLflow supports **nested runs** via `nested=True`:

```python
with mlflow.start_run(run_name="sweep-parent") as parent:
    for C in [0.1, 1.0, 10.0]:
        with mlflow.start_run(run_name=f"C={C}", nested=True) as child:
            mlflow.log_param("C", C)
            # ... train and log metrics ...
```

In the UI, child runs are collapsed under the parent row and can be expanded. This keeps the run list clean when doing dozens of sweeps.

In [ ]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-classification")

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
results = []

# Parent run holds the sweep configuration
with mlflow.start_run(run_name="C-sweep-parent") as parent:
    mlflow.log_param("sweep_param", "C")
    mlflow.log_param("sweep_values", str(C_values))
    mlflow.set_tag("run_type", "sweep")

    for C in C_values:
        # nested=True links this run to the parent
        with mlflow.start_run(run_name=f"LR-C={C}", nested=True) as child:
            mlflow.log_params({"C": C, "solver": "lbfgs", "max_iter": 200})
            mlflow.set_tag("model_type", "logistic_regression")

            clf = LogisticRegression(C=C, max_iter=200, solver="lbfgs")
            clf.fit(X_train, y_train)
            acc = accuracy_score(y_test, clf.predict(X_test))
            mlflow.log_metric("test_accuracy", acc)

            results.append({"C": C, "test_accuracy": acc, "child_run_id": child.info.run_id[:8]})

    # Log the best C on the parent run for easy retrieval
    best = max(results, key=lambda r: r["test_accuracy"])
    mlflow.log_metric("best_test_accuracy", best["test_accuracy"])
    mlflow.log_param("best_C", best["C"])

    print(f"Parent run: {parent.info.run_id[:8]}")
    print("\nChild run results:")
    for r in results:
        marker = " <-- best" if r["C"] == best["C"] else ""
        print(f"  C={r['C']:6.2f}  acc={r['test_accuracy']:.4f}  id={r['child_run_id']}{marker}")

### What just happened?
- **`nested=True`** creates a parent-child link visible in the UI's run tree.
- The **parent run** stores sweep-level metadata; each **child run** stores its own params and metrics.
- Logging `best_C` and `best_test_accuracy` on the parent makes it easy to retrieve the winner with a single `get_run(parent_id)` call.
- This pattern scales to hundreds of child runs — the UI remains readable because children are collapsed.

In [ ]:
# Challenge: Multi-model experiment sweep
#
# Instructions:
#   1. Create a new experiment named "iris-model-comparison"
#   2. Run a sweep over at least 3 different model types
#      (e.g. LogisticRegression, RandomForest, KNeighborsClassifier)
#   3. For each model:
#      - Log model_type as a tag
#      - Log at least one hyperparameter
#      - Log test_accuracy
#   4. Use search_runs to print the models ranked by test_accuracy
#   5. Bonus: wrap the sweep in a parent run using nested=True
#
# Scaffold:

# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression

# mlflow.set_experiment("iris-model-comparison")

# models = [
#     {"name": "knn",    "model": KNeighborsClassifier(n_neighbors=5), "params": {"n_neighbors": 5}},
#     {"name": ???,      "model": ???,                                  "params": ???},
#     {"name": ???,      "model": ???,                                  "params": ???},
# ]

# for m in models:
#     with mlflow.start_run(run_name=m["name"]):
#         # ... your code here ...
#         pass

# Your solution here


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `set_experiment` | Always call at the top of every script; creates if missing |
| `start_run` / `end_run` | Use `with` for safety; explicit for multi-cell notebooks |
| `log_metric(..., step=N)` | Creates a time-series; visible as a chart in the UI |
| `set_tag` / `set_tags` | Free-form string metadata; mutable after run ends |
| `search_runs` (module) | Returns a DataFrame; filter with SQL-like `filter_string` |
| `nested=True` | Groups child runs under a parent; keeps UI clean during sweeps |
| `filter_string` syntax | `metrics.X > N`, `tags.X = 'val'`, `AND` / `OR` |

> **Tip:** Use `mlflow.set_experiment()` at the top of every script — orphaned runs in the Default experiment are hard to find later.

---
## What's next
**Day 3** → Go beyond metrics and params: log trained sklearn models, Matplotlib figures, Pandas DataFrames, and entire artifact directories — and navigate the artifact browser to verify them.

Mark Day 2 complete in your [tracker](../index.html).